# YOLO26l Tree Detection — Colab Training

Trains YOLO26l on GDINO pseudo-labeled video frames.
- freeze=5, epochs=100, batch=auto
- imgsz=1280, rect=True, mosaic=0.5

**Drive 目录结构 (MyDrive/TreeLearn/):**
```
TreeLearn/
  images/train/*.jpg   (521 files)
  images/val/*.jpg     (522 files)
  labels/train/*.txt   (521 files)
  labels/val/*.txt     (522 files)
  train_yolo26l_colab.ipynb
```


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.4.56

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# If you have two 'TreeLearn' folders, pick the one rclone created
# (the one containing images/ and labels/ subfolders directly)
DRIVE_DIR = Path('/content/drive/MyDrive/TreeLearn')
DATA_DIR  = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Verify structure
for p in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    d = DRIVE_DIR / p
    n = len(list(d.glob('*'))) if d.exists() else -1
    print(f'{p}: {n} files  [{"OK" if n > 0 else "MISSING"}]')

In [ ]:
# Copy dataset from Drive to local SSD (much faster I/O during training)
import shutil

print('Copying dataset to local SSD...')
for split in ('train', 'val'):
    for kind in ('images', 'labels'):
        src = DRIVE_DIR / kind / split
        dst = DATA_DIR  / kind / split
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n = len(list(dst.glob('*')))
        print(f'  {kind}/{split}: {n} files')

print('Done.')

In [ ]:
import yaml

data_yaml = {
    'path': str(DATA_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'nc': 1,
    'names': ['tree'],
}
yaml_path = DATA_DIR / 'data.yaml'
yaml_path.write_text(yaml.dump(data_yaml))
print(yaml_path.read_text())

In [ ]:
import torch
from ultralytics import YOLO
from ultralytics.utils.autobatch import check_train_batch_size

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB')

m = YOLO('yolo26l.pt')
m.model = m.model.cuda()
BATCH = check_train_batch_size(m.model, imgsz=1280, amp=True)
del m; torch.cuda.empty_cache()
print(f'Using batch={BATCH}')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26l.pt')
model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=1280,
    batch=BATCH,
    freeze=5,
    optimizer='auto',
    lr0=1e-3,
    lrf=0.01,
    cos_lr=True,
    patience=20,
    save=True,
    project='/content/runs/detect',
    name='tree_yolo26l_gdino',
    mosaic=0.5,
    mixup=0.0,
    degrees=10.0,
    translate=0.1,
    scale=0.3,
    flipud=0.0,
    fliplr=0.5,
    device=0,
    workers=4,
    cache=True,
    rect=True,
    deterministic=False,
)

In [ ]:
import shutil
from pathlib import Path

run_dir = Path('/content/runs/detect/tree_yolo26l_gdino')
dst = DRIVE_DIR / 'weights'
dst.mkdir(parents=True, exist_ok=True)
for fname in ('best.pt', 'last.pt'):
    shutil.copy2(run_dir / 'weights' / fname, dst / fname)
shutil.copy2(run_dir / 'results.csv', dst / 'results.csv')
print(f'Saved weights + results.csv → {dst}')